In [1]:
# Cell 1: Import required libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta


In [2]:
# Cell 2: Create realistic power consumption data
np.random.seed(42)

# Generate timestamps: 7 days, readings every 5 minutes (288/day)
start_date = datetime(2025, 5, 12, 0, 0, 0)
minutes_per_day = 24 * 60
interval_minutes = 5
readings_per_day = minutes_per_day // interval_minutes  # 288
total_days = 7
total_readings = readings_per_day * total_days  # 2,016

# Create timestamp array
timestamps = [start_date + timedelta(minutes=i*interval_minutes) 
              for i in range(total_readings)]

# Generate power consumption (Watts) with realistic daily pattern
# Peak during day (9 AM - 5 PM), low at night
hour_of_day = [ts.hour + ts.minute/60 for ts in timestamps]

# Base load: 500W at night, increasing during working hours
base_load = 500 + 1500 * np.sin(np.pi * (np.array(hour_of_day) - 6) / 12)
base_load = np.maximum(base_load, 500)  # Minimum 500W

# Add random variations
noise = np.random.normal(0, 100, total_readings)

# Add occasional spikes (machines starting up)
spikes = np.random.exponential(200, total_readings) * (np.random.random(total_readings) < 0.02)

power_watts = base_load + noise + spikes
power_watts = np.maximum(power_watts, 0)  # No negative power

# Create DataFrame
df = pd.DataFrame({
    'timestamp': timestamps,
    'power_watts': power_watts
})

print(f"📊 Generated {len(df):,} readings over {total_days} days")
print(f"   First reading: {df['timestamp'].iloc[0]}")
print(f"   Last reading: {df['timestamp'].iloc[-1]}")

📊 Generated 2,016 readings over 7 days
   First reading: 2025-05-12 00:00:00
   Last reading: 2025-05-18 23:55:00


In [3]:
# Cell 3: View first and last few rows
print("First 10 readings:")
print(df.head(10))
print("\nLast 5 readings:")
print(df.tail(5))
print(f"\nDataFrame shape: {df.shape}")
print(f"Data types:\n{df.dtypes}")

First 10 readings:
            timestamp  power_watts
0 2025-05-12 00:00:00   549.671415
1 2025-05-12 00:05:00   486.173570
2 2025-05-12 00:10:00   564.768854
3 2025-05-12 00:15:00   652.302986
4 2025-05-12 00:20:00   476.584663
5 2025-05-12 00:25:00   476.586304
6 2025-05-12 00:30:00   657.921282
7 2025-05-12 00:35:00   576.743473
8 2025-05-12 00:40:00   453.052561
9 2025-05-12 00:45:00   554.256004

Last 5 readings:
               timestamp  power_watts
2011 2025-05-18 23:35:00   423.727522
2012 2025-05-18 23:40:00   423.085765
2013 2025-05-18 23:45:00   406.009690
2014 2025-05-18 23:50:00   582.947484
2015 2025-05-18 23:55:00   480.617386

DataFrame shape: (2016, 2)
Data types:
timestamp      datetime64[ns]
power_watts           float64
dtype: object


In [4]:
# Cell 4: Convert power (Watts) to energy (kWh)
# Energy (kWh) = Power (kW) × Time (hours)
# Time between readings = 5 minutes = 5/60 = 0.08333 hours

interval_hours = interval_minutes / 60  # 0.08333 hours

# Convert Watts to kW and multiply by time
df['power_kw'] = df['power_watts'] / 1000
df['energy_kwh'] = df['power_kw'] * interval_hours

print(f"⚡ Added energy column (kWh per {interval_minutes}-minute interval)")
print(f"   Sample energy values (first 5):")
print(df[['timestamp', 'power_watts', 'power_kw', 'energy_kwh']].head())

⚡ Added energy column (kWh per 5-minute interval)
   Sample energy values (first 5):
            timestamp  power_watts  power_kw  energy_kwh
0 2025-05-12 00:00:00   549.671415  0.549671    0.045806
1 2025-05-12 00:05:00   486.173570  0.486174    0.040514
2 2025-05-12 00:10:00   564.768854  0.564769    0.047064
3 2025-05-12 00:15:00   652.302986  0.652303    0.054359
4 2025-05-12 00:20:00   476.584663  0.476585    0.039715


In [5]:
# Cell 5: Create date and hour columns for grouping
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()

print("📅 Added time components:")
print(df[['timestamp', 'date', 'hour', 'day_of_week']].head())

📅 Added time components:
            timestamp        date  hour day_of_week
0 2025-05-12 00:00:00  2025-05-12     0      Monday
1 2025-05-12 00:05:00  2025-05-12     0      Monday
2 2025-05-12 00:10:00  2025-05-12     0      Monday
3 2025-05-12 00:15:00  2025-05-12     0      Monday
4 2025-05-12 00:20:00  2025-05-12     0      Monday


In [7]:
# Cell 6: Use groupby to calculate total energy per day
daily_energy = df.groupby('date')['energy_kwh'].sum().reset_index()
daily_energy.columns = ['Date', 'Total_Energy_kWh']

print("📊 DAILY ENERGY CONSUMPTION:")
print("-" * 45)
print(daily_energy.to_string(index=False))
print("-" * 45)
print(f"Total 7-day energy: {daily_energy['Total_Energy_kWh'].sum():.2f} kWh")
print(f"Average daily energy: {daily_energy['Total_Energy_kWh'].mean():.2f} kWh")

📊 DAILY ENERGY CONSUMPTION:
---------------------------------------------
      Date  Total_Energy_kWh
2025-05-12         23.499325
2025-05-13         23.413187
2025-05-14         23.618475
2025-05-15         23.840448
2025-05-16         23.707272
2025-05-17         23.782052
2025-05-18         23.525048
---------------------------------------------
Total 7-day energy: 165.39 kWh
Average daily energy: 23.63 kWh


In [8]:
# Cell 7: Find the hour with highest average power each day
# (Peak demand = hour when grid sees highest load)

def find_peak_hour(day_data):
    """Find the hour with maximum average power for a given day"""
    hourly_avg = day_data.groupby('hour')['power_watts'].mean()
    peak_hour = hourly_avg.idxmax()
    peak_power = hourly_avg.max()
    return peak_hour, peak_power

# Calculate peak hour for each day
peak_results = []
for date in df['date'].unique():
    day_data = df[df['date'] == date]
    peak_hour, peak_power = find_peak_hour(day_data)
    peak_results.append({
        'Date': date,
        'Peak_Hour': f"{peak_hour:02d}:00 - {peak_hour+1:02d}:00",
        'Peak_Power_W': round(peak_power, 0)
    })

peak_df = pd.DataFrame(peak_results)

print("\n⚡ PEAK DEMAND HOURS BY DAY:")
print("-" * 55)
print(peak_df.to_string(index=False))


⚡ PEAK DEMAND HOURS BY DAY:
-------------------------------------------------------
      Date     Peak_Hour  Peak_Power_W
2025-05-12 12:00 - 13:00        1977.0
2025-05-13 11:00 - 12:00        2000.0
2025-05-14 11:00 - 12:00        1986.0
2025-05-15 11:00 - 12:00        2037.0
2025-05-16 12:00 - 13:00        1987.0
2025-05-17 11:00 - 12:00        1962.0
2025-05-18 12:00 - 13:00        1991.0


In [9]:
# Cell 8: Group by hour across all days to see daily pattern
hourly_stats = df.groupby('hour').agg({
    'power_watts': ['mean', 'max', 'min'],
    'energy_kwh': 'sum'
}).round(2)

hourly_stats.columns = ['Avg_Power_W', 'Max_Power_W', 'Min_Power_W', 'Total_Energy_kWh_7days']

print("\n📈 HOURLY STATISTICS (across 7 days):")
print(hourly_stats.head(12))  # First 12 hours


📈 HOURLY STATISTICS (across 7 days):
      Avg_Power_W  Max_Power_W  Min_Power_W  Total_Energy_kWh_7days
hour                                                               
0          515.67       729.09       250.06                    3.61
1          515.70       797.29       229.68                    3.61
2          519.07       709.24       316.94                    3.63
3          505.92       744.58       304.03                    3.54
4          495.15       755.82       323.70                    3.47
5          500.07       758.22       230.31                    3.50
6          685.23      1050.61       303.45                    4.80
7         1058.22      1818.24       797.85                    7.41
8         1408.44      1702.00      1046.13                    9.86
9         1668.25      2003.32      1389.35                   11.68
10        1885.26      2660.96      1588.09                   13.20
11        1983.82      2240.38      1740.70                   13.89


In [10]:
# Cell 9: Merge daily energy with peak hours
daily_report = daily_energy.merge(peak_df, on='Date')
daily_report['Energy_kWh'] = daily_report['Total_Energy_kWh'].round(2)
daily_report = daily_report[['Date', 'Energy_kWh', 'Peak_Hour', 'Peak_Power_W']]

print("\n" + "="*70)
print("📋 ENERGY AUDIT DAILY REPORT")
print("="*70)
print(daily_report.to_string(index=False))
print("="*70)

# Find highest energy day and highest peak day
max_energy_day = daily_report.loc[daily_report['Energy_kWh'].idxmax()]
max_peak_day = daily_report.loc[daily_report['Peak_Power_W'].idxmax()]

print(f"\n💡 INSIGHTS:")
print(f"   Highest energy day: {max_energy_day['Date']} ({max_energy_day['Energy_kWh']} kWh)")
print(f"   Highest peak demand: {max_peak_day['Date']} ({max_peak_day['Peak_Power_W']:.0f} W at {max_peak_day['Peak_Hour']})")


📋 ENERGY AUDIT DAILY REPORT
      Date  Energy_kWh     Peak_Hour  Peak_Power_W
2025-05-12       23.50 12:00 - 13:00        1977.0
2025-05-13       23.41 11:00 - 12:00        2000.0
2025-05-14       23.62 11:00 - 12:00        1986.0
2025-05-15       23.84 11:00 - 12:00        2037.0
2025-05-16       23.71 12:00 - 13:00        1987.0
2025-05-17       23.78 11:00 - 12:00        1962.0
2025-05-18       23.53 12:00 - 13:00        1991.0

💡 INSIGHTS:
   Highest energy day: 2025-05-15 (23.84 kWh)
   Highest peak demand: 2025-05-15 (2037 W at 11:00 - 12:00)


In [11]:
# Cell 10: Export results to CSV files
daily_report.to_csv("energy_audit_daily_report.csv", index=False)
peak_df.to_csv("peak_demand_hours.csv", index=False)
hourly_stats.to_csv("hourly_statistics.csv")

print("💾 Reports saved:")
print("   - energy_audit_daily_report.csv (daily totals)")
print("   - peak_demand_hours.csv (peak hours per day)")
print("   - hourly_statistics.csv (hourly patterns)")

💾 Reports saved:
   - energy_audit_daily_report.csv (daily totals)
   - peak_demand_hours.csv (peak hours per day)
   - hourly_statistics.csv (hourly patterns)
